In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [44]:
episode_df = pd.read_csv(r"dataset\fineTuned\top10_episodes.csv",index_col=0)
embeddings = np.load(r"dataset\notFineTuned\embeddings\balanced_episode_embeddings.npy")
episode_df['published_date'] = pd.to_datetime(episode_df['published_date'])
episode_df = episode_df.reset_index()

In [45]:
print("Number of sentences:", len(episode_df))
episode_df.head()

Number of sentences: 337


,orig_index,content_id,sentence,published_date,category,timestamp,episode,t
0,0,1442420,"Until now, the AirPods Pro were all about keep...",2024-09-17,Gadgets,2024-09-17,18,208
1,1,1442420,Apple says the latest AirPods Pro 2 can be use...,2024-09-17,Gadgets,2024-09-17,18,208
2,3,1442420,The company is also planning to integrate a he...,2024-09-17,Gadgets,2024-09-17,18,208
3,5,1442420,At Apple's annual September product launch eve...,2024-09-17,Gadgets,2024-09-17,18,208
4,7,1442420,The new AirPods also have a slightly modified ...,2024-09-17,Gadgets,2024-09-17,18,208


In [46]:
indices = episode_df["orig_index"].values
episode_embeddings = embeddings[indices]

## Clustering within episode to find events

In [47]:
import hdbscan

In [48]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=4,
    metric='euclidean',
    prediction_data=True
) 

event_labels = clusterer.fit_predict(episode_embeddings)

c:\Users\gaura\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\gaura\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [49]:
episode_df["event_id"] = event_labels

In [50]:
print("Unique event IDs:", set(event_labels))
print("Number of clusters (excluding -1):", len(set(event_labels)) - (1 if -1 in event_labels else 0))
print("Noise points:", list(event_labels).count(-1))

Unique event IDs: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, -1}
Number of clusters (excluding -1): 10
Noise points: 31


In [51]:
episode_df = episode_df[episode_df["event_id"] != -1].copy()

In [52]:
event_counts = episode_df["event_id"].value_counts()

print(event_counts)

4    95
0    35
8    35
6    31
2    25
1    21
3    20
7    15
9    15
5    14
Name: event_id, dtype: int64


In [53]:
valid_events = event_counts[event_counts >= 3].index

episode_df = episode_df[episode_df["event_id"].isin(valid_events)].copy()
episode_df.shape

(306, 9)

In [54]:
episode_df = episode_df.sort_values(by=["event_id", "published_date"])

In [55]:
for event in episode_df["event_id"].unique():
    print("\n======================")
    print(f"Event ID: {event}")
    print("======================")
    
    samples = episode_df[episode_df["event_id"] == event]["sentence"].head(5)
    
    for s in samples:
        print("-", s)


Event ID: 0
- Children are developing nearsightedness earlier in life, and while looking too long and too close at phones and tablets may play a role, the answer is not that simple, according to a Connecticut paediatric ophthalmologist in the US.
- “We are seeing an increased rate of myopia over the last, I would say, 20 years or so, and it is projected to increase,” said Dr Majida Gaffar, division head of ophthalmology at Connecticut Children’s Medical Center.
- “It’s definitely something that ophthalmologists and optometrists are looking at to slow the Myopia affects about 5% of preschoolers, 9% of school-aged children and 30% of teens, according to While there is no cause for myopia, the formal name for nearsightedness, there are correlations, such as “near work and diet ... environmental factors,” Gaffar said.
- Also, the child of someone with myopia is more likely to be nearsighted.
- “I’ll always tell my patients to limit as much as possible, even though they haven’t really foun

## Pair-wise dataset preparation for fine-tuning

In [56]:
grouped = episode_df.groupby("event_id")

### Positive pair (Label:1)

In [62]:
from itertools import combinations
import random

In [63]:
positive_pairs = []

for event_id, group in grouped:
    sentences = group["sentence"].tolist()
    
    for s1, s2 in combinations(sentences, 2):
        positive_pairs.append((s1, s2, 1))

print(len(positive_pairs))

7121


In [65]:
for i in range(3):
    pair_id = random.randint(0,len(positive_pairs))
    print("\n======================")
    print(f"Pair {pair_id}:")
    print("======================")
    print(positive_pairs[pair_id][0])
    print(positive_pairs[pair_id][1])


Pair 1238:
When asked why those in the community bother with going through the hassle of building keyboards by themselves instead of going with something ready made off-the-shelf, Row likened the hobby to car enthusiasts.
Some people have keyboards just for more technical work like video editing or coding.

Pair 6064:
And Apple has offered its in-house Translate app on the iPhone since 2020.
The capability will work like this: If an English speaker is hearing someone talk in Spanish, the iPhone will translate the speech and relay it to the user’s AirPods in English.

Pair 49:
“We are seeing an increased rate of myopia over the last, I would say, 20 years or so, and it is projected to increase,” said Dr Majida Gaffar, division head of ophthalmology at Connecticut Children’s Medical Center.
While there is such a thing as pathologic myopia, which would be a prescription of 6 diopters or more, that is not common, Gaffar said.


### Negative pairs (Label:0)

In [68]:
negative_pairs = []

event_ids = list(grouped.groups.keys())

num_neg = len(positive_pairs)

for _ in range(num_neg):
    e1, e2 = random.sample(event_ids, 2)
    
    s1 = random.choice(grouped.get_group(e1)["sentence"].tolist())
    s2 = random.choice(grouped.get_group(e2)["sentence"].tolist())
    
    negative_pairs.append((s1, s2, 0))

print(len(negative_pairs))

7121


In [66]:
for i in range(3):
    pair_id = random.randint(0,len(negative_pairs))
    print("\n======================")
    print(f"Pair {pair_id}:")
    print("======================")
    print(negative_pairs[pair_id][0])
    print(negative_pairs[pair_id][1])


Pair 6580:
The new law, which passed by a 61-1 vote in the Colorado House and a 34-0 vote in the Senate, expands the definition of “sensitive data” in the state’s current personal privacy law to include biological and “neural data” generated by the brain, the spinal cord and the network of nerves that relays messages throughout the body.
“And so when kids come in with really high myopia, the eye is so long and you can imagine that the retina is a bit stretched out, it can cause thinning of the retina and retinal detachments.

Pair 5896:
The entry prices for each product remain the same, with the iPad Pro starting at US$999 (from RM4,499 locally), the MacBook Pro coming in at US$1,599 (from RM6,999 locally), and the Vision Pro headset remaining US$3,499 (RM14,792).
“Airbnb said the inside is States vary on whether and what degree of consent is required for surveillance, and there are different rules for audio and video recording.

Pair 7118:
All three devices look identical to the curr

### Combining the positive and negative pair

In [70]:
all_pairs = positive_pairs + negative_pairs

pairs_df = pd.DataFrame(all_pairs, columns=["sentence1", "sentence2", "label"])
pairs_df = pairs_df.sample(frac=1).reset_index(drop=True)

print(pairs_df.shape)
pairs_df.head()

(14242, 3)


,sentence1,sentence2,label
0,Level up with 5G While current tools like sens...,The cost of upgrading hardware to be 5G-ready ...,1
1,The role played by 5G in this scenario comes l...,"“The keyboard is a modern, essential tool for ...",0
2,The 5G coverage in populated areas in the coun...,We also face a people barrier with farmers not...,1
3,"But with 5G, one operator can manage multiple ...","""MCMC has started conducting test-bed sessions...",1
4,Extra storage pricing is similar to Apple’s at...,If you find a hidden camera in a hotel room or...,0


In [71]:
pairs_df.to_csv(r"dataset\fineTuned\pairwise_dataset.csv", index=False)